In [ ]:
class TradingEnv:

    def __init__(self, feature_df, initial_cash=10000):

        self.data = feature_df
        self.prices = feature_df["price"]

        self.initial_cash = initial_cash
        self.current_step = 0

        self.cash = initial_cash
        self.shares_held = 0
        self.portfolio_value = initial_cash

        self.transaction_cost = 0.001
        self.entry_price = None

        self.valid_buys = 0
        self.invalid_buys = 0

        self.trade_profits = []

    def reset(self):

        self.current_step = 0
        self.cash = self.initial_cash
        self.shares_held = 0
        self.portfolio_value = self.initial_cash
        self.entry_price = None
        self.trade_profits = []

        return self.get_state()

    def get_state(self):

        row = self.data.iloc[self.current_step]
    
        current_price = self.prices.iloc[self.current_step]
    
        if self.entry_price is None:
            unrealized_pnl = 0.0
        else:
            unrealized_pnl = (
                current_price - self.entry_price
            ) / self.entry_price
    
        if self.portfolio_value > 0:
            position_fraction = (
                self.shares_held * current_price
            ) / self.portfolio_value
        else:
            position_fraction = 0.0
    
        return [
    
            row["momentum"],
            row["volatility"],
            row["ma_signal"],
            row["rsi"],
    
            row["return_20"],
            row["return_50"],
    
            row["ma50_signal"],
            row["ma200_signal"],
    
            row["atr_proxy"],
    
            unrealized_pnl,
            position_fraction
        ]

    def step(self, action):

        current_price = self.prices.iloc[self.current_step]

        reward_penalty = 0
        reward_bonus = 0

        old_portfolio_value = self.portfolio_value

        if action == 1:

            fraction = 0.5

            if self.cash >= current_price:

                amount_invest = self.cash * fraction

                shares_to_buy = (
                    amount_invest / current_price
                )

                fee = (
                    amount_invest
                    * self.transaction_cost
                )

                self.cash -= (
                    amount_invest + fee
                )

                self.shares_held += shares_to_buy

                if self.entry_price is None:
                    self.entry_price = current_price

            else:

                reward_penalty = -0.001

        elif action == 2:

            fraction = 1.0

            if self.cash >= current_price:

                amount_invest = self.cash * fraction

                shares_to_buy = (
                    amount_invest / current_price
                )

                fee = (
                    amount_invest
                    * self.transaction_cost
                )

                self.cash -= (
                    amount_invest + fee
                )

                self.shares_held += shares_to_buy

                if self.entry_price is None:
                    self.entry_price = current_price

            else:

                reward_penalty = -0.001

        elif action == 3:

            fraction = 0.5

            if self.shares_held > 0:

                shares_to_sell = self.shares_held * fraction
                sale_val = shares_to_sell* current_price

                fee = sale_val* self.transaction_cost

                self.cash += (sale_val - fee)
                profit_pct = (current_price- self.entry_price) / self.entry_price

                reward_bonus = profit_pct * 10

                self.trade_profits.append(profit_pct)
                self.shares_held -= shares_to_sell

                if self.shares_held <= 1e-8:

                    self.shares_held = 0
                    self.entry_price = None

            else:

                reward_penalty = -0.005

        elif action == 4:

            fraction = 1.0

            if self.shares_held > 0:
                shares_to_sell = self.shares_held * fraction
                sale_val = shares_to_sell* current_price

                fee = sale_val* self.transaction_cost

                self.cash += (sale_val - fee)

                profit_pct = (current_price- self.entry_price) / self.entry_price

                reward_bonus = (profit_pct)

                self.trade_profits.append(profit_pct)
                self.shares_held -= shares_to_sell

                if self.shares_held <= 1e-8:

                    self.shares_held = 0
                    self.entry_price = None

            else:

                reward_penalty = -0.005

        self.current_step += 1

        done = (self.current_step>= len(self.prices) - 1)

        new_price = self.prices.iloc[self.current_step]

        self.portfolio_value = (self.cash+ self.shares_held * new_price)
        reward = self.portfolio_value
            

        #reward += reward_penalty
        #reward += reward_bonus

        next_state = self.get_state()

        return next_state, reward, done